# Predictive Anayltics: Support Vector Machines with Regression

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [52]:
#TODO: Änderungen von Classification übernehmen -> sollte gemacht sein
#TODO: embedding ansehen -> vielleicht austauschen

In [53]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 10_000
SPATIAL_UNIT = "HEXAGON" # options: CENSUS_TRACTS, HEXAGON, COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4H" # options: 1H, 2H, 4H
H3_RES = "8" # options 7,8

In [54]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [55]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder
import pytorch_lightning as py_light

from sklearn.preprocessing import OneHotEncoder

import networkx as nx
from libpysal.weights import Queen
from node2vec import Node2Vec

## Preparations

In [56]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [57]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TRAIN.parquet"
DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TEST.parquet"
DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"
    DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"
    DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
]

Load data and select features and target

In [58]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [59]:
#keep_cols = [
#    SPATIAL_UNIT,  # "h3_cell"
#    "trip_count",  
#    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
#    "hour_sin", "hour_cos",
#    "tmpc", "relh", "sknt", "vsby", "p01m",
#    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
#    "is_holiday",
#    "food_drink", "landmark", "shop", "train_station",
#]

In [60]:
#keep_cols = np.array(keep_cols)

#if SPATIAL_UNIT == "HEXAGON":
#    keep_cols = np.append(keep_cols,"h3_cell")
#elif SPATIAL_UNIT == "CENSUS_TRACTS":
#    keep_cols = np.append(keep_cols,"community_area")
#elif SPATIAL_UNIT == "COMMUNITY_AREAS":
#    keep_cols = np.append(keep_cols,"community_area")
#    print("here")

In [61]:
#train_df = train[keep_cols].copy()
#test_df = test[keep_cols].copy()
#val_df = val[keep_cols].copy()

In [62]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

val_df["food_drink"] = val_df["food_drink"].fillna(0.0)
val_df["landmark"] = val_df["landmark"].fillna(0.0)
val_df["shop"] = val_df["shop"].fillna(0.0)
val_df["train_station"] = val_df["train_station"].fillna(0.0)

test_df["food_drink"] = test_df["food_drink"].fillna(0.0)
test_df["landmark"] = test_df["landmark"].fillna(0.0)
test_df["shop"] = test_df["shop"].fillna(0.0)
test_df["train_station"] = test_df["train_station"].fillna(0.0)

In [63]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-04-07 08:00:00,4,2,8,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-08-21 00:00:00,8,4,0,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-21 20:00:00,8,4,20,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2026-04-17 04:00:00,4,5,4,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-03-15 16:00:00,3,6,16,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [64]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [65]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [66]:
# encode into lat long


if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "HEXAGON":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, val_df, test_df):
            df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

    elif SPATIAL_UNIT == "CENSUS_TRACTS":
        print("Encoding: latlong and Unit: census_tract")

        # load census tract
        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, test_df):
            df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
            df["lat"] = df["census_tract"].map(tract_centroids["lat"])
            df["lon"] = df["census_tract"].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

    elif SPATIAL_UNIT == "COMMUNITY_AREAS":
        print("Encoding: latlong and Unit: community_area")

        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  

        gdf["lon"] = gdf.geometry.centroid.x
        gdf["lat"] = gdf.geometry.centroid.y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(2)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")


Encoding: latlong and Unit: hexa


In [67]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"].values, df["lon"].values)  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Onehot

In [68]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
   # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)

### Spatial Encoding: Spatial Embedding

In [69]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "HEXAGON"):

    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])

    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Features per H3 cell (mean of POI features, train only)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Required by srai: maps each region to its features
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]

elif (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "CENSUS_TRACTS"):
    census_data = gpd.read_file(CENSUS_PATH)
    census_data["geometry"] = gpd.GeoSeries.from_wkt(census_data["the_geom"])

    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")

    gdf["TRACT_FIPS"] = gdf["TRACT_FIPS"].astype(str)
    gdf = gdf.set_index("TRACT_FIPS")

    gdf = gdf.to_crs(epsg=5070)
    gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid]
    gdf = gdf[~gdf.index.duplicated(keep="first")]

    w = Queen.from_dataframe(gdf, use_index=True)
    G = w.to_networkx()
    print(f"Graph: {G.number_of_nodes()} tracts, {G.number_of_edges()} adjacency edges")

    node2vec = Node2Vec(
        G,
        dimensions=32,
        walk_length=20,
        num_walks=100,
        workers=4,
        p=1,
        q=1,
    )

    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    embedding_dict = {node: model.wv[node] for node in G.nodes()}
    emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")

    emb_cols = [f"emb_{i}" for i in range(emb_df.shape[1])]
    emb_df.columns = emb_cols

    train_df = train_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    val_df = val_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    test_df = test_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


Create y

In [70]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [71]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,...,skyc1_SCT,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station,x,y,z
0,1.000000,6.123234e-17,0.781831,0.623490,0.866025,-0.5,-0.417500,55.720000,11.500000,10.000000,...,0,0,0,2.0,0.0,1.0,0.0,0.029151,-0.742671,0.669022
1,-0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.0,22.288000,67.930000,9.400000,10.000000,...,1,0,0,3.0,0.0,5.0,2.0,0.030940,-0.745371,0.665932
2,-0.500000,-8.660254e-01,0.433884,-0.900969,-0.866025,0.5,24.442500,58.212500,8.000000,10.000000,...,1,0,0,3.0,0.0,5.0,2.0,0.030940,-0.745371,0.665932
3,1.000000,6.123234e-17,-0.433884,-0.900969,0.866025,0.5,9.784615,98.655385,4.307692,0.538462,...,0,0,0,3.0,0.0,5.0,2.0,0.030940,-0.745371,0.665932
4,0.866025,5.000000e-01,-0.974928,-0.222521,-0.866025,-0.5,18.910000,41.064000,21.400000,6.400000,...,0,0,0,0.0,0.0,0.0,0.0,0.031331,-0.744567,0.666812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373609,0.500000,8.660254e-01,-0.781831,0.623490,0.866025,0.5,-3.984286,78.674286,8.714286,5.928571,...,0,0,0,1.0,2.0,2.0,4.0,0.029796,-0.744259,0.667226
373610,0.000000,1.000000e+00,0.433884,-0.900969,0.000000,1.0,-6.807500,52.685000,16.750000,10.000000,...,0,0,0,1.0,2.0,2.0,4.0,0.029796,-0.744259,0.667226
373611,0.866025,5.000000e-01,0.000000,1.000000,0.866025,-0.5,2.500000,60.855000,14.000000,10.000000,...,0,0,0,0.0,0.0,0.0,2.0,0.027494,-0.742766,0.668987
373612,0.000000,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,-8.330000,58.632000,16.200000,10.000000,...,0,0,0,0.0,0.0,0.0,2.0,0.027494,-0.742766,0.668987


In [72]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val_grid = scaler.fit_transform(X_val_grid)

### Grid Search

In [73]:
model = SVR()

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.020599212918366788 best params: {'C': 10, 'epsilon': 1.5, 'kernel': 'linear'}
rbf_sigmoid best score: 0.09677232944810406 best params: {'C': 10, 'epsilon': 1.5, 'gamma': 0.01, 'kernel': 'rbf'}


In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 1, 'degree': 3, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}
Best CV score: 0.7796365575375757


In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",1
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 6.06623197,  1.34785082,  0.92840365, ..., -3.8927612 ,
       -0.86106653, -1.00588888])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 16.578284127035385
MSE: 4093.5482175240877
RMSE: 63.98084258216742
R2 Score: 0.790151368685712


In [ ]:
# save model
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']